In [1]:
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks

In [2]:
PROJECT_ROOT = Path.cwd()

# Если ноутбук открыт из папки notebooks/, поднимаемся в корень проекта
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
OUTPUTS = PROJECT_ROOT / "outputs"
ANNOTATIONS_DIR = OUTPUTS / "annotations"
FIGURES_DIR = OUTPUTS / "figures"

for folder in [DATA_RAW, ANNOTATIONS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("Outputs folder:", OUTPUTS)

Project root: /Users/user/Documents/emg-lr-segmentation
Raw data folder: /Users/user/Documents/emg-lr-segmentation/data/raw
Outputs folder: /Users/user/Documents/emg-lr-segmentation/outputs


In [4]:
fif_path = DATA_RAW / "3_3_rhythm_late_responses_raw.fif"

assert fif_path.exists(), f"Файл не найден: {fif_path}"

raw = mne.io.read_raw_fif(fif_path, preload=True)
raw_original = raw.copy()
raw_work = raw.copy()

print(raw)
print(raw.ch_names)
print("sfreq:", raw.info["sfreq"])

Opening raw data file /Users/user/Documents/emg-lr-segmentation/data/raw/3_3_rhythm_late_responses_raw.fif...
Isotrak not found
    Range : 0 ... 164899 =      0.000 ...    41.225 secs
Ready.
Reading 0 ... 164899  =      0.000 ...    41.225 secs...
<Raw | 3_3_rhythm_late_responses_raw.fif, 7 x 164900 (41.2 s), ~8.8 MiB, data loaded>
['EMG1', 'EMG2', 'EMG3', 'EMG4', 'EMG5', 'EMG6', 'EMG7']
sfreq: 4000.0


In [5]:
raw.plot(duration=10, n_channels=16)

Using qt as 2D backend.


Channels marked as bad:
none


In [4]:
# Сохраняем разметку в файл прямо рядом с вашим кодом
raw.annotations.save("lr_annotations_31_05_2026.csv", overwrite=True)
print(raw.annotations)

Overwriting existing file.
<Annotations | 1322 segments: ER (23), LR (30), Stimulus (22), ...>


In [ ]:
events, event_id = mne.events_from_annotations(raw)
print("Все найденные метки:", event_id)

tmin = 0.0
tmax = 0.6 

epochs = mne.Epochs(
    raw, 
    events=events, 
    event_id=event_id['Stimulus'], # Режем ТОЛЬКО по этой метке
    tmin=tmin, 
    tmax=tmax, 
    preload=True,
    baseline=None # Отключаем коррекцию, если просто смотрим сигнал
)

Used Annotations descriptions: [np.str_('Stimulus')]
Все найденные метки: {np.str_('Stimulus'): 1}
Not setting metadata
9 matching events found


No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 9 events and 8001 original time points ...
0 bad epochs dropped


In [ ]:
# 1. Извлекаем данные канала 'Art'
art_data, times = raw.copy().pick(['Art']).get_data(return_times=True)
art_signal = np.abs(art_data[0])
sfreq = raw.info['sfreq']

# Находим первый ручной стимул
user_stimulus = [a for a in raw.annotations if a['description'] == 'Stimulus']
if not user_stimulus:
    raise ValueError("Пожалуйста, разметьте самый первый 'Stimulus' вручную!")
    
user_stimulus = sorted(user_stimulus, key=lambda x: x['onset'])
first_stim_onset = user_stimulus[0]['onset']
ideal_duration = user_stimulus[0]['duration']

# Параметры сетки
step_sec = 0.033  # Базовый шаг
search_window_sec = 0.006  # Окно поиска физического пика вокруг расчетной точки (6 мс)
search_window_samples = int(search_window_sec * sfreq)

raw_duration = raw.times[-1]
all_stim_onsets = [first_stim_onset]
current_onset = first_stim_onset

# 2. Пошагово генерируем метки с динамической коррекцией по каналу Art
while True:
    # Делаем математический шаг вперед
    expected_onset = current_onset + step_sec
    if expected_onset >= raw_duration:
        break
        
    # Переводим ожидаемое время в индекс сэмпла
    expected_sample = np.searchsorted(times, expected_onset)
    
    # Определяем границы окрестности для поиска реального пика
    start_idx = max(0, expected_sample - search_window_samples)
    end_idx = min(len(art_signal), expected_sample + search_window_samples)
    
    # Ищем самый сильный всплеск на канале Art в этом узком окошке
    local_window = art_signal[start_idx:end_idx]
    
    if len(local_window) > 0 and np.max(local_window) > np.mean(art_signal) * 3:
        # Если реальный пик найден — привязываемся строго к его вершине!
        actual_peak_idx = start_idx + np.argmax(local_window)
        current_onset = times[actual_peak_idx]
    else:
        # Если пика нет (пропуск в записи), берем математическое ожидание
        current_onset = expected_onset
        
    all_stim_onsets.append(current_onset)

print(f"📈 Сгенерировано {len(all_stim_onsets)} адаптивных меток.")

# 3. Записываем скорректированные аннотации в MNE
stim_durations = [ideal_duration] * len(all_stim_onsets)
stim_descriptions = ['Stimulus_Auto'] * len(all_stim_onsets)

auto_stim_annots = mne.Annotations(
    onset=all_stim_onsets,
    duration=stim_durations,
    description=stim_descriptions,
    orig_time=raw.annotations.orig_time
)

clean_user_annotations = mne.Annotations(
    onset=[a['onset'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    duration=[a['duration'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    description=[a['description'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    orig_time=raw.annotations.orig_time
)

raw.set_annotations(clean_user_annotations + auto_stim_annots)
print("✅ Готово! Сетка стимулов успешно адаптирована под дрейф времени прибора.")
raw.plot()

🛠️ Запуск адаптивной авторазметки с защитой от съезжания...
📈 Сгенерировано 1247 адаптивных меток.
✅ Готово! Сетка стимулов успешно адаптирована под дрейф времени прибора.


qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


Using pyopengl with version 3.1.9


<mne_qt_browser._pg_figure.MNEQtBrowser(0x7f8e86af6c10) at 0x1af3164c0>

Channels marked as bad:
[np.str_('Art')]


In [92]:

# 1. Конвертируем текстовые аннотации Stimulus_Auto в события MNE (Events)
events, event_id = mne.events_from_annotations(raw, regexp='Stimulus_Auto')

# 2. Задаем временное окно эпохи (от стимула до стимула)
tmin = 0.0     # Начало эпохи (0 мс — сам момент стимула)
tmax = 0.033    # Конец эпохи (330 мс — аккурат перед следующим стимулом)

# 3. Нарезаем эпохи
# baseline=None означает, что мы пока не вычитаем среднее пре-стимульное значение
# preload=True загружает данные в оперативную память для быстрой работы
epochs = mne.Epochs(
    raw, 
    events=events, 
    event_id=event_id, 
    tmin=tmin, 
    tmax=tmax, 
    baseline=None, 
    preload=True
)

print(f"🎯 Успешно нарезано эпох: {len(epochs)}")
print(f"📐 Форма массива данных эпох (epochs, channels, times): {epochs.get_data().shape}")

Used Annotations descriptions: [np.str_('Stimulus_Auto')]
Not setting metadata
1247 matching events found
No baseline correction applied


0 projection items activated
Using data from preloaded Raw for 1247 events and 133 original time points ...
1 bad epochs dropped
🎯 Успешно нарезано эпох: 1246
📐 Форма массива данных эпох (epochs, channels, times): (1246, 7, 133)


In [56]:
epochs = mne.Epochs(
    raw, 
    events=filtered_stim_events, 
    event_id=stim_id,
    tmin=0.0, 
    tmax=epoch_duration,          
    picks=['GM R'],  # Оставляем только нужный канал            
    baseline=None,
    preload=True
)

# Защита от ошибок np.int64 для словаря цветов в MNE
labels_to_show = {k: v for k, v in event_id.items() if k in ['ER', 'LR']}
custom_colors = {
    np.int64(er_id): 'green',  
    np.int64(lr_id): 'blue',   
    np.int64(stim_id): 'lightgray'
}

print("🚀 Запуск интерактивного окна MNE...")
print("📌 ВАЖНО: Закройте открывшееся окно графиков, чтобы разблокировать блокнот для следующей ячейки!")

# Открываем встроенный браузер MNE
epochs.plot(
    n_epochs=10,               
    n_channels=1,              
    scalings='auto',      
    events=events,             
    event_id=labels_to_show, 
    event_color=custom_colors,
    picks='all',
    block=True  # Ждем, пока вы закроете окно, прежде чем выполнять код дальше
)

Not setting metadata
10 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 10 events and 133 original time points ...
0 bad epochs dropped
🚀 Запуск интерактивного окна MNE...
📌 ВАЖНО: Закройте открывшееся окно графиков, чтобы разблокировать блокнот для следующей ячейки!


qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


Using pyopengl with version 3.1.9
Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


<mne_qt_browser._pg_figure.MNEQtBrowser(0x0) at 0x199be5040>

In [ ]:
# 1. Находим индексы эпох, где есть И волна ER, И волна LR
valid_indices = []
for i, meta in enumerate(valid_meta):
    # Проверяем, что списки ER и LR внутри этой эпохи не пусты
    if meta['ER'] and meta['LR']:
        valid_indices.append(i)

print(f"🔍 Всего в записи найдено {len(valid_indices)} эпох, содержащих одновременно ER и LR.")

# 2. Берем первые 10 эпох (или сколько есть, если вдруг меньше 10)
indices_to_plot = valid_indices[:10]

if len(indices_to_plot) == 0:
    print("Не найдено ни одной эпохи, где одновременно размечены и ER, и LR. Проверьте названия меток.")
else:
    # 3. Строим сетку графиков 2x5 или сколько необходимо
    n_plots = len(indices_to_plot)
    ncols = 5
    nrows = int(np.ceil(n_plots / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows), sharex=True, sharey=True)
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    # Получаем данные отфильтрованных эпох (для красивого гладкого отображения)
    epochs_data = epochs_filtered.get_data(picks=['GM R'])[:, 0, :]
    times_ms = epochs_filtered.times * 1000
    
    for plot_idx, epoch_idx in enumerate(indices_to_plot):
        ax = axes[plot_idx]
        signal = epochs_data[epoch_idx] * 1e6  # Переводим в мкВ
        
        ax.plot(times_ms, signal, color='black', lw=1.2, label='Signal')
        
        # Подсвечиваем зоны ER (зеленым) и LR (синим) прямо из вашей ручной разметки
        meta = valid_meta[epoch_idx]
        stim_onset = meta['onset']
        
        # Рисуем ER
        for er in meta['ER']:
            start_ms = (er['onset'] - stim_onset) * 1000
            end_ms = start_ms + (er['duration'] * 1000)
            ax.axvspan(start_ms, end_ms, color='green', alpha=0.2, label='ER' if plot_idx==0 else "")
            
        # Рисуем LR
        for lr in meta['LR']:
            start_ms = (lr['onset'] - stim_onset) * 1000
            end_ms = start_ms + (lr['duration'] * 1000)
            ax.axvspan(start_ms, end_ms, color='blue', alpha=0.2, label='LR' if plot_idx==0 else "")
            
        ax.grid(True, linestyle=':', alpha=0.5)
        
    # Удаляем пустые окошки, если эпох оказалось меньше, чем ячеек в сетке
    for j in range(plot_idx + 1, len(axes)):
        fig.delaxes(axes[j])
        
    # Добавляем общую легенду и подписи
    axes[0].legend(fontsize=8, loc='upper right')
    fig.text(0.5, 0.01, 'Time from Stimulus (ms)', ha='center', fontsize=11)
    fig.text(0.01, 0.5, r'Amplitude ($\mu$V)', va='center', rotation='vertical', fontsize=11)
    plt.suptitle("GM R Channel", fontsize=12, weight='bold', y=0.98)
    plt.tight_layout()
    plt.show()